<a href="https://colab.research.google.com/github/Glaze0/Assignment/blob/main/RAG_w_PDF(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pypdf faiss-cpu sentence-transformers google-genai requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 30.6 MB/s eta 0:00:00


In [2]:
# ============================================================
# BASIC PDF RAG — COMPLETE EXAMPLE
# PDF → Text → Chunks → Embeddings → FAISS → Gemini
# ============================================================

import requests
import faiss
import numpy as np

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google import genai


# ============================================================
# 1. SETUP
# ============================================================

client = genai.Client(
    api_key="__",
)

MODEL = "gemini-3.5-flash"

embedder = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# ============================================================
# 2. DOWNLOAD PDF
# ============================================================

pdf_url = (
    "https://raw.githubusercontent.com/harvard-hbs/"
    "rag-example/main/source_documents/"
    "5008_Federalist%20Papers.pdf"
)

pdf_path = "federalist_papers.pdf"

response = requests.get(pdf_url)

with open(pdf_path, "wb") as f:
    f.write(response.content)

print("PDF downloaded successfully.")


# ============================================================
# 3. READ FIRST X PAGES
# ============================================================

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages[:3]:

    page_text = page.extract_text()

    if page_text:
        text += page_text + "\n"

print("Pages read:", min(1, len(reader.pages)))
print("Characters extracted:", len(text))


# ============================================================
# 4. SPLIT TEXT INTO CHUNKS
# ============================================================

def create_chunks(text, chunk_size=800):

    chunks = []

    for i in range(0, len(text), chunk_size):

        chunk = text[i:i + chunk_size]

        if chunk.strip():
            chunks.append(chunk)

    return chunks


chunks = create_chunks(text)

print("Number of chunks:", len(chunks))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

PDF downloaded successfully.
Pages read: 1
Characters extracted: 10840
Number of chunks: 14


In [3]:
# ============================================================
# 5. CREATE EMBEDDINGS
# ============================================================

embeddings = embedder.encode(
    chunks,
    normalize_embeddings=True
).astype("float32")

print("Embedding shape:", embeddings.shape)


# ============================================================
# 6. CREATE FAISS INDEX
# ============================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)    #used to index the chunk using dimension

index.add(embeddings)

print("Vectors stored in FAISS:", index.ntotal)

Embedding shape: (14, 384)
Vectors stored in FAISS: 14


In [4]:
# ============================================================
# 7. RETRIEVE RELEVANT CHUNKS
# ============================================================

def retrieve(query, k=3):

    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    k = min(k, len(chunks))

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for i in indices[0]:
        results.append(chunks[i])

    return results


# ============================================================
# 8. BASIC RAG
# ============================================================

def basic_rag(query):

    # -------------------------
    # RETRIEVE
    # -------------------------

    relevant_chunks = retrieve(
        query,
        k=3
    )

    context = "\n\n---\n\n".join(
        relevant_chunks
    )


    # -------------------------
    # AUGMENT
    # -------------------------

    prompt = f"""
You are a helpful assistant.

Answer the question using ONLY the
provided document context.

If the answer is not available in
the context, say:

"I don't know from this document."

DOCUMENT CONTEXT:
{context}

QUESTION:
{query}
"""


    # -------------------------
    # GENERATE
    # -------------------------

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response.text

In [ ]:
# ============================================================
# 9. TEST THE RAG
# ============================================================

question = input(
    "\nAsk a question about the PDF: "
)

answer = basic_rag(question)

print("\nAnswer:")
print(answer)